In [ ]:
!pip install autogluon.tabular ucimlrepo google-adk

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, accuracy_score, confusion_matrix, f1_score
from ucimlrepo import fetch_ucirepo
from autogluon.tabular import TabularDataset, TabularPredictor

mi_data = fetch_ucirepo(id=579)
X_full = mi_data.data.features.copy()
y_full = mi_data.data.targets.copy()
y_full = y_full.fillna(y_full.mode().iloc[0])
target_names = [y_full.columns.tolist()[0]] 
binary_targets = target_names

def get_admission_data(X_data):
    day_1_cols = ['R_AB_1_n', 'NA_R_1_n', 'NOT_NA_1_n']
    day_2_cols = ['R_AB_2_n', 'NA_R_2_n', 'NOT_NA_2_n']
    day_3_cols = ['R_AB_3_n', 'NA_R_3_n', 'NOT_NA_3_n']
    drop_cols = day_1_cols + day_2_cols + day_3_cols
    return X_data.drop(columns=[c for c in drop_cols if c in X_data.columns])

X_adm = get_admission_data(X_full)
df_full = pd.concat([X_adm, y_full], axis=1)

train_data, test_data = train_test_split(df_full, test_size=0.2, random_state=42)
train_data = TabularDataset(train_data)
test_data = TabularDataset(test_data)

predictors = {}
predicted_train_features = train_data.copy()
predicted_test_features = test_data.copy()

print(f"\nTraining AutoGluon for Single Target: {target_names[0]}...")
PENALTY_FACTOR = 5.0 
target = target_names[0]

In [ ]:
num_pos = (predicted_train_features[target] == 1).sum()
num_neg = (predicted_train_features[target] == 0).sum()
weight_ratio = (num_neg / (num_pos + 1e-5)) * PENALTY_FACTOR 

predicted_train_features['sample_weight'] = np.where(predicted_train_features[target] == 1, weight_ratio, 1.0)
weight_col = 'sample_weight'

predictor = TabularPredictor(
    label=target, 
    eval_metric='roc_auc',
    problem_type='binary',
    sample_weight=weight_col
).fit(
    predicted_train_features, 
    presets='good_quality', 
    time_limit=180, 
    verbosity=0 
)

predictors[target] = predictor